# XGBoost — Optuna Tuning

## Imports

In [ ]:
import optuna
import numpy as np
import os
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

work_dir = os.getcwd()
print(work_dir)

## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_normalized")
X = np.load(os.path.join(input_folder, "X_train.npy"))
y = np.load(os.path.join(input_folder, "y_train.npy"))

groups = np.load(os.path.join(work_dir, "data_group", "groups_train.npy"), allow_pickle=True)

X = X.reshape(X.shape[0], -1)   # flatten for XGBoost
print(f"X={X.shape}  y={y.shape}  groups={groups.shape}")

## Objective

In [ ]:
def objective(trial):
    config = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 2000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 1, 10),
        "gamma"            : trial.suggest_float("gamma", 0, 5),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 0.5, 2),
        "tree_method"      : "hist",
        "random_state"     : 42,
        "eval_metric"      : "rmse",
        "early_stopping_rounds": 50,
    }

    gkf   = GroupKFold(n_splits=3)
    rmses = []
    for _, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_v = X[tr_idx], X[val_idx]
        y_tr, y_v = y[tr_idx], y[val_idx]
        m = XGBRegressor(**config)
        m.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y_v, m.predict(X_v))))
    return np.mean(rmses)


## Study

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

## Results

In [ ]:
import matplotlib.pyplot as plt
print('Best RMSE  :', study.best_value)
print('Best Params:', study.best_params)

optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout(); plt.show()
optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout(); plt.show()
optuna.visualization.matplotlib.plot_slice(study)
plt.tight_layout(); plt.show()